# Vector Memory Demo

This notebook exercises the new reusable vector retrieval layer and the memory-specific adapter.

It covers:
- generic vector backend contract
- memory indexing via `MemoryIndexer`
- hybrid retrieval via `HybridMemoryRetriever`
- config flags for vector memory search
- a future-proof shape that can later back a RAG node

This demo uses `SentenceTransformerEmbeddingProvider` with `sentence-transformers/all-MiniLM-L6-v2`.
Install it with `pip install -e ".[memory-vector-local]"` before running the notebook.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from tempfile import TemporaryDirectory


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo_root={REPO_ROOT}")


repo_root=/Users/saketm10/Projects/openclaw_agents


In [2]:
from src.memory.index import MemoryIndexer, VectorMemoryIndexBackend
from src.memory.layers import ColdMemoryLayer, HotMemoryLayer, WarmMemoryLayer
from src.memory.retrieval.hybrid_retriever import HybridMemoryRetriever
from src.memory.store import MemoryStore
from src.memory.types import SemanticMemory
from src.retrieval.sentence_transformer_embedding import SentenceTransformerEmbeddingProvider
from src.retrieval.models import IndexedItem, RetrievalHit
from src.retrieval.vector_backend import VectorRetrievalBackend
from src.utils.config import AppSettings


class InMemoryVectorBackend(VectorRetrievalBackend):
    """Minimal backend implementing the shared vector contract for notebook testing."""

    def __init__(self) -> None:
        self.items: dict[str, IndexedItem] = {}

    def upsert(self, item: IndexedItem) -> None:
        self.items[item.item_id] = item

    def delete(self, item_id: str) -> None:
        self.items.pop(item_id, None)

    def search(self, vector: list[float], filters: dict[str, object] | None = None, limit: int = 20) -> list[RetrievalHit]:
        filters = filters or {}
        hits: list[RetrievalHit] = []
        for item in self.items.values():
            if filters.get("scope") and item.metadata.get("scope") != filters["scope"]:
                continue
            if filters.get("agent_id") and item.metadata.get("agent_id") != filters["agent_id"]:
                continue
            if filters.get("type") and item.metadata.get("type") != filters["type"]:
                continue
            score = sum(left * right for left, right in zip(vector, item.vector, strict=False))
            hits.append(RetrievalHit(item_id=item.item_id, score=score, text=item.text, metadata=item.metadata))
        return sorted(hits, key=lambda hit: hit.score, reverse=True)[:limit]

    def rebuild(self, items: list[IndexedItem]) -> None:
        self.items = {item.item_id: item for item in items}


def build_store(root: Path) -> MemoryStore:
    return MemoryStore(
        hot_layer=HotMemoryLayer(max_items=8),
        warm_layer=WarmMemoryLayer(root / "memory.duckdb"),
        cold_layer=ColdMemoryLayer(root / "cold.jsonl"),
        archive_after_days=30,
        default_scope="agent_local",
        agent_id="mailmind",
    )


tmp = TemporaryDirectory()
DEMO_ROOT = Path(tmp.name)
store = build_store(DEMO_ROOT)
backend = InMemoryVectorBackend()
provider = SentenceTransformerEmbeddingProvider(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
# Fallback for zero-dependency local testing:
# from src.retrieval.hash_embedding import HashEmbeddingProvider
# provider = HashEmbeddingProvider(_dimension=32)
index_backend = VectorMemoryIndexBackend(backend)
indexer = MemoryIndexer(embedding_provider=provider, index_backend=index_backend)
retriever = HybridMemoryRetriever(store=store, embedding_provider=provider, index_backend=index_backend, vector_top_k=5)

print(f"demo_root={DEMO_ROOT}")


demo_root=/var/folders/x3/r36gzhhj13b6sbqsg2hgn0zm0000gn/T/tmp6r8br1fe


In [3]:
# Seed a few semantic memories.
exact = SemanticMemory(content={"fact": "DeepMind research role"}).store_warm(store)
semantic = SemanticMemory(content={"fact": "research lab opportunity"}).store_warm(store)
other = SemanticMemory(content={"fact": "weekly finance newsletter"}).store_warm(store)

print(exact.id, exact.content_text)
print(semantic.id, semantic.content_text)
print(other.id, other.content_text)


2712e8ed-0cfc-491d-82ed-b5939bfa8932 {'fact': 'DeepMind research role'}
6a5bbba9-6190-462b-86b0-c8b0dcae26c7 {'fact': 'research lab opportunity'}
12f92696-caf5-4241-831b-fdc181312034 {'fact': 'weekly finance newsletter'}


In [4]:
# Index the memories through the memory-specific adapter.
indexed = indexer.index_records([exact, semantic, other])

print("indexed_records:")
for item in indexed:
    print(item.model_dump(mode="json"))

print("\nraw_backend_items:")
for item_id, item in backend.items.items():
    print(item_id, item.metadata)


indexed_records:
{'record_id': '2712e8ed-0cfc-491d-82ed-b5939bfa8932', 'text': "{'fact': 'DeepMind research role'}", 'metadata': {'scope': 'agent_local', 'agent_id': 'mailmind', 'type': 'semantic', 'layer': 'warm', 'tags': [], 'source_type': 'system', 'source_id': None}}
{'record_id': '6a5bbba9-6190-462b-86b0-c8b0dcae26c7', 'text': "{'fact': 'research lab opportunity'}", 'metadata': {'scope': 'agent_local', 'agent_id': 'mailmind', 'type': 'semantic', 'layer': 'warm', 'tags': [], 'source_type': 'system', 'source_id': None}}
{'record_id': '12f92696-caf5-4241-831b-fdc181312034', 'text': "{'fact': 'weekly finance newsletter'}", 'metadata': {'scope': 'agent_local', 'agent_id': 'mailmind', 'type': 'semantic', 'layer': 'warm', 'tags': [], 'source_type': 'system', 'source_id': None}}

raw_backend_items:
2712e8ed-0cfc-491d-82ed-b5939bfa8932 {'scope': 'agent_local', 'agent_id': 'mailmind', 'type': 'semantic', 'layer': 'warm', 'tags': [], 'source_type': 'system', 'source_id': None}
6a5bbba9-6190-

In [5]:
# Run hybrid retrieval. This merges warm-store keyword matches with vector similarity hits.
results = retriever.retrieve(
    "DeepMind research",
    filters={"type": "semantic", "agent_id": "mailmind"},
    limit=5,
)

print("hybrid_results:")
for record in results:
    print(record.id, record.content_text)


hybrid_results:
2712e8ed-0cfc-491d-82ed-b5939bfa8932 {'fact': 'DeepMind research role'}
6a5bbba9-6190-462b-86b0-c8b0dcae26c7 {'fact': 'research lab opportunity'}
12f92696-caf5-4241-831b-fdc181312034 {'fact': 'weekly finance newsletter'}


In [6]:
# Compare with plain store search.
keyword_results = store.search(
    query="DeepMind research",
    filters={"type": "semantic", "agent_id": "mailmind"},
    limit=5,
)

print("keyword_results:")
for record in keyword_results:
    print(record.id, record.content_text)


keyword_results:
2712e8ed-0cfc-491d-82ed-b5939bfa8932 {'fact': 'DeepMind research role'}


In [7]:
# Directly exercise the generic vector backend contract.
query_vector = provider.embed_text("research opportunity")
vector_hits = backend.search(query_vector, filters={"type": "semantic", "agent_id": "mailmind"}, limit=5)

print("vector_hits:")
for hit in vector_hits:
    print(hit.model_dump(mode="json"))


vector_hits:
{'item_id': '2712e8ed-0cfc-491d-82ed-b5939bfa8932', 'score': 0.0, 'text': "{'fact': 'DeepMind research role'}", 'metadata': {'scope': 'agent_local', 'agent_id': 'mailmind', 'type': 'semantic', 'layer': 'warm', 'tags': [], 'source_type': 'system', 'source_id': None}}
{'item_id': '6a5bbba9-6190-462b-86b0-c8b0dcae26c7', 'score': 0.0, 'text': "{'fact': 'research lab opportunity'}", 'metadata': {'scope': 'agent_local', 'agent_id': 'mailmind', 'type': 'semantic', 'layer': 'warm', 'tags': [], 'source_type': 'system', 'source_id': None}}
{'item_id': '12f92696-caf5-4241-831b-fdc181312034', 'score': 0.0, 'text': "{'fact': 'weekly finance newsletter'}", 'metadata': {'scope': 'agent_local', 'agent_id': 'mailmind', 'type': 'semantic', 'layer': 'warm', 'tags': [], 'source_type': 'system', 'source_id': None}}


In [8]:
# Show the current config flags that enable this feature path.
settings = AppSettings.from_env()
print(settings.memory.model_dump(mode="json"))

print("\nRelevant env vars:")
print("EASY_AGENT_MEMORY_SIMILARITY_ENABLED")
print("EASY_AGENT_MEMORY_SIMILARITY_BACKEND")
print("EASY_AGENT_MEMORY_EMBEDDING_PROVIDER")
print("EASY_AGENT_MEMORY_EMBEDDING_MODEL")
print("EASY_AGENT_MEMORY_HYBRID_SEARCH_ENABLED")
print("EASY_AGENT_MEMORY_VECTOR_TOP_K")


{'hot_cache_size': 256, 'archive_after_days': 30, 'escalation_step_count': 3, 'confidence_threshold': 0.5, 'similarity_enabled': False, 'similarity_backend': 'none', 'embedding_provider': 'hash', 'embedding_model': 'hash-64', 'hybrid_search_enabled': False, 'vector_top_k': 20}

Relevant env vars:
EASY_AGENT_MEMORY_SIMILARITY_ENABLED
EASY_AGENT_MEMORY_SIMILARITY_BACKEND
EASY_AGENT_MEMORY_EMBEDDING_PROVIDER
EASY_AGENT_MEMORY_EMBEDDING_MODEL
EASY_AGENT_MEMORY_HYBRID_SEARCH_ENABLED
EASY_AGENT_MEMORY_VECTOR_TOP_K


In [9]:
# Why this is reusable for a future RAG node:
# - the generic contract is VectorRetrievalBackend
# - memory indexing adapts MemoryRecord -> IndexedItem
# - a future RAG node can adapt document chunks -> IndexedItem using the same backend

print("Reusable layers:")
print("- src/retrieval/* is generic")
print("- src/memory/index/* is memory-specific")
print("- a future RAG node should use src/retrieval/* plus its own chunk/document adapter")


Reusable layers:
- src/retrieval/* is generic
- src/memory/index/* is memory-specific
- a future RAG node should use src/retrieval/* plus its own chunk/document adapter
